In [52]:
import numpy as np
import cv2
import pandas as pd
import matplotlib.pyplot as plt

from typing import Any, Optional, Tuple, Generic, TypeVar
from collections import namedtuple

BBox = namedtuple("BBox", "x_min y_min width height")
Shape = TypeVar("Shape")
DType = TypeVar("DType")

class Frame(np.ndarray, Generic[Shape, DType]):
    pass

In [53]:
class CF:
    def __init__(self, sigma: float = 1.0, num_perturbations: int = 256, min_psr: float = 8.0, seed: int = 2137):
        self._sigma = sigma
        self._num_perturbations = num_perturbations
        self._min_psr = min_psr
        self._bbox_xywh = None
        np.random.seed(seed)

    def init(self, frame: np.ndarray, bbox_xywh) -> bool:
        assert frame.ndim == 2 and frame.dtype == np.uint8, "invalid frame"
        self._bbox_xywh = BBox(*bbox_xywh)
        assert self._bbox_xywh.width > 0 and self._bbox_xywh.height > 0, "invalid bbox"

        x, y, w, h = bbox_xywh
        roi = frame[y:y + h, x:x + w]
        roi = self._preprocess(roi)

        self._Gi = np.fft.fft2(self._get_gauss(w, h))
        fi = self._perturbate_frame(roi, self._num_perturbations)

        Fi = np.fft.fft2(fi, axes=(1, 2))
        Fi_conj = np.conj(Fi)

        Gi_rep = np.repeat(self._Gi[None, :, :], self._num_perturbations, axis=0)
        Num = Gi_rep * Fi_conj
        Denom = Fi * Fi_conj

        self._Num = np.sum(Num, axis=0)
        self._Denom = np.sum(Denom, axis=0)
        return True

    def update(self, frame: np.ndarray, rate: float = 0.07, eps: float = 1e-5) -> BBox:
        assert frame.ndim == 2 and frame.dtype == np.uint8, "invalid frame"
        x, y, w, h = self._bbox_xywh
        roi = frame[y:y + h, x:x + w]
        fi = self._preprocess(roi)
        Fi = np.fft.fft2(fi)
        Hi = self._Num / (self._Denom + eps)
        response = np.fft.ifft2(Hi * Fi).real

        max_loc = np.unravel_index(np.argmax(response), response.shape)
        dy, dx = max_loc[0] - h // 2, max_loc[1] - w // 2

        new_bbox = self.correct_bbox(BBox(x + dx, y + dy, w, h), frame.shape[1], frame.shape[0])
        if new_bbox.width != w or new_bbox.height != h:
            self.init(frame, new_bbox)
        else:
            self._update_filter(frame, self._bbox_xywh, rate)
        self._bbox_xywh = new_bbox

        # PSR calculation
        mean = np.mean(response)
        std = np.std(response) + eps
        psr = (response[max_loc] - mean) / std
        return new_bbox

    def _get_gauss(self, width: int, height: int) -> np.ndarray:
        x_center = width / 2
        y_center = height / 2
        yy, xx = np.meshgrid(np.arange(height), np.arange(width), indexing="ij")
        gauss = np.exp(-((xx - x_center) ** 2 + (yy - y_center) ** 2) / (2 * self._sigma))
        gauss -= gauss.min()
        gauss /= (gauss.max() - gauss.min() + 1e-8)
        return gauss

    def _update_filter(self, frame: np.ndarray, bbox: BBox, rate: float):
        x, y, w, h = bbox
        roi = frame[y:y + h, x:x + w]
        fi = self._preprocess(roi)
        Fi = np.fft.fft2(fi)
        Fi_conj = np.conj(Fi)

        self._Num = (1 - rate) * self._Num + rate * self._Gi * Fi_conj
        self._Denom = (1 - rate) * self._Denom + rate * Fi * Fi_conj

    def _perturbate_frame(self, frame: np.ndarray, num_samples: int, degree: float = 18.0) -> np.ndarray:
        h, w = frame.shape
        center = (w // 2, h // 2)
        samples = []

        for _ in range(num_samples):
            angle = np.random.uniform(-degree, degree)
            rot_mat = cv2.getRotationMatrix2D(center, angle, 1.0)
            rotated = cv2.warpAffine(frame, rot_mat, (w, h), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REFLECT)
            samples.append(rotated)

        return np.stack(samples, axis=0)

    def _preprocess(self, frame: np.ndarray, eps: float = 1e-5) -> np.ndarray:
        frame = np.log(frame.astype(np.float32) + 1)
        frame = (frame - frame.mean()) / (np.linalg.norm(frame) + eps)
        return frame * self.hann(*frame.shape)

    @staticmethod
    def hann(height: int, width: int) -> np.ndarray:
        hann1 = np.hanning(height)
        hann2 = np.hanning(width)
        return np.outer(hann1, hann2)

    @staticmethod
    def correct_bbox(bbox: BBox, width: int, height: int) -> BBox:
        x, y, w, h = bbox
        x = max(0, min(x, width - 1))
        y = max(0, min(y, height - 1))
        w = min(w, width - x)
        h = min(h, height - y)
        return BBox(x, y, w, h)

In [54]:
def bbox_iou(box1, box2):

    # Transform from center and width to exact coordinates
    b1_x1, b1_x2 = box1[0], box1[0] + box1[2]
    b1_y1, b1_y2 = box1[1], box1[1] + box1[3]
    b2_x1, b2_x2 = box2[0], box2[0] + box2[2]
    b2_y1, b2_y2 = box2[1], box2[1] + box2[3]

    # get the corrdinates of the intersection rectangle
    inter_rect_x1 = max(b1_x1, b2_x1)
    inter_rect_y1 = max(b1_y1, b2_y1)
    inter_rect_x2 = min(b1_x2, b2_x2)
    inter_rect_y2 = min(b1_y2, b2_y2)
    # Intersection area
    inter_area = np.clip(inter_rect_x2 - inter_rect_x1, a_min=0, a_max=None) * np.clip(inter_rect_y2 - inter_rect_y1, a_min=0, a_max=None)
    # Union Area
    b1_area = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    b2_area = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)

    iou = inter_area / (b1_area + b2_area - inter_area + 1e-16)

    return iou

In [55]:
path = 'sequences/jump/'
ious_per_sequence = {}

groundtruth = pd.read_csv(path+"/groundtruth.txt").to_numpy(int)

frame1 = cv2.imread(path+"color/00000001.jpg", cv2.IMREAD_GRAYSCALE)
filter = CF(sigma=1, num_perturbations=1024)
filter.init(frame1, groundtruth[0])
ious = []

for i in range(1, 172):
    frame = cv2.imread(f"{path}color/{i:08d}.jpg", cv2.IMREAD_GRAYSCALE)
    res_box = filter.update(frame)

    frame = cv2.rectangle(frame, (res_box[0], res_box[1]), (res_box[2]+res_box[0], res_box[3]+res_box[1]), (255, 0, 0), 2)
    cv2.imshow("Frame", frame)
    if cv2.waitKey(50) == ord('q'):
        cv2.destroyAllWindows()
        break

    iou = bbox_iou(res_box, groundtruth[i-1])
    ious.append(iou)

ious_per_sequence = np.mean(ious)
print("IOUS : ", np.mean(ious))
cv2.destroyAllWindows()

IOUS :  0.6274825994198993
